# Breakout 4B · Informality: matching customs and tax

Workshop **Data Driven Domestic Revenue Mobilization**, Bangkok, October 2026. Co-run with the tax facilitator: every result says which side acts.

The session moves left to right along three tiers of data. Every exercise starts in the first two;
every "what it takes" names the third as the thing the executive has to sign for.

| Tier | What it is | How you get it | What it unlocks here |
| --- | --- | --- | --- |
| **Open data** | Published by statistical offices, registers, international organisations. Anyone can download it today. | A browser and a licence check. | A register to match against (track B), the shape of borrowed-name networks, the informality framing, the method for counting informal trade. |
| **Your own data** | Declarations, the importer register, control outcomes, the tax register and returns of the same administration. | An extract from the clearance system and the tax system; a read-only replica for anything monthly. | The exact and probable match, the four buckets *if* the two registers share an identifier, the five patterns. |
| **Shared data** | What another administration or a partner holds: the tax side's returns for customs, the customs declarations for tax, a neighbour's export statistics, beneficial-ownership and broker registers. | A legal basis, a shared identifier, a return flow, cost-sharing, named contact points. | The buckets with a lead agency, the joint visit, the borrowed-name case that needs the beneficial owner, the land-border mirror. |

**How to use it.** Press ▶ on each cell from top to bottom, or *Runtime › Run all*. Change only the sliders and drop-down menus. The code is folded: double-click a title to see it. Every result ends with a **Finding** sentence.

Everything in sections 1, 2, 4 and 5 is synthetic. Section 3 runs on a real public register: its rows
are called by `reg_id` and `cust_id` only, and no name from it is ever printed.

In [ ]:
#@title 0 · Where the data comes from
#@markdown Leave these as they are. Your analysts can point DATA_URL at their own extract (section 6).
DATA_URL = "https://raw.githubusercontent.com/FrancoisChastel/CustomsWorkshopNotebooks/data-v7/data"  #@param {type:"string"}
INSTALL = True  #@param {type:"boolean"}

In [ ]:
#@title 0 · Setup (run me first)
import sys, subprocess, warnings
warnings.filterwarnings("ignore")
if INSTALL and "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "git+https://github.com/FrancoisChastel/CustomsWorkshopNotebooks@data-v7"], check=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from workshop_methods import notebook as nb
from workshop_methods.data import load_json, load_tables
from workshop_methods import border, informality
nb.use_style()
pd.set_option("display.max_colwidth", 40)
CALIBRATION = "random draw finds something on 1 to 8 lines in a hundred; this set is enriched on the 20 exercise lines only."
def calibration_line():
    """Printed under every table that shows a rate."""
    display(Markdown(f"<small><i>Calibration: {CALIBRATION}</i></small>"))
HOME_USE = ("home_use", "warehouse_out")  # the regimes where duty is paid: the lines a price check ranks
T = load_tables(DATA_URL, ['importers', 'tax_registry', 'vat_returns', 'importers_4b', 'match_example_600', 'features_lines', 'tariff', 'parcels', 'land_border_mirror', 'trader_features', 'brokers'])
print(f"Loaded {len(T)} tables from {DATA_URL}. Ready.")

In [ ]:
#@title 0 · Prepare the tables
importers, registry, returns, f, tf = T["importers"], T["tax_registry"], T["vat_returns"], T["features_lines"], T["trader_features"].set_index("importer_id")
importers = importers.assign(tin=importers["tin"].fillna(""))
periods = sorted(returns["period"].unique())
sheet = T["importers_4b"].rename(columns={"tin_customs": "tin", "importer_name": "legal_name", "import_value_12m_lcu": "import_value_12m"})
sheet["tin"] = sheet["tin"].fillna("")
print(f"{len(importers):,} importers on the customs register ({importers['tin'].eq('').sum()} without a tax ID); the tax registry holds "
      f"{len(registry):,} taxpayers with {len(periods)} periods of returns. The exercise sheet holds {len(sheet)} importers.")

## 1 · Exact match: join on the tax ID

In [ ]:
#@title 1 · Join on the tax ID and explain every miss
nb.steps("How many importers can we find in the tax registry by their tax ID, and why do the others miss?",
         "Join each importer's tax ID, as customs holds it, to the registry.",
         "Give the reason for every miss: no ID on the customs side, an ID one character off, or an ID the registry does not hold.",
         "Read the same join on the 600-importer example of the deck.")
exact = informality.exact_match(importers, registry)
counts = exact["exact_status"].value_counts().rename("importers").to_frame()
counts["share"] = counts["importers"] / len(exact)
counts["import_value_12m"] = exact.groupby("exact_status")["importer_id"].apply(lambda s: tf["value_12m_lcu"].reindex(s).sum())
nb.show(counts, formats={"share": "{:.0%}", "import_value_12m": "{:,.0f}"})
six = exact[exact["importer_id"].isin(T["match_example_600"]["importer_id"])]["exact_status"].value_counts()
in_registry = int(six.get("matched", 0) + six.get("TIN one character off", 0))
nb.finding(f"{counts.at['matched', 'importers']:,} of {len(exact):,} importers match on the tax ID; {int(counts['importers'].drop('matched').sum())} miss, "
           f"{int(counts['importers'].get('blank TIN', 0))} because customs holds no ID and {int(counts['importers'].get('TIN one character off', 0))} because "
           f"it holds one a character off. On the deck's 600-importer example: {in_registry} are in the registry ({int(six.get('matched', 0))} exact, "
           f"{int(six.get('TIN one character off', 0))} to be found by name, phone and director) and {int(six.get('blank TIN', 0))} have no link at all. "
           "An exact join is right when it matches and silent when it does not: validate IDs at registration with a check digit.")

## 2 · Probable match and the dial

In [ ]:
#@title 2 · Match on name, address, phone and director, and set the dial
PRESET = "balanced"  #@param ["strict", "balanced", "loose", "use the slider"]
THRESHOLD = 85  #@param {type:"slider", min:50, max:100, step:5}
threshold = {"strict": 95, "balanced": 85, "loose": 70}.get(PRESET, THRESHOLD)
nb.steps("When the ID fails, can names, addresses, phones and directors find the taxpayer, and how strict should we be?",
         "Clean the names (lower case, legal suffixes removed); compare only pairs with the same name start and district.",
         "Score name similarity (0 to 100, weighted 0.7) plus 10 points each for the same address, phone and director.",
         f"Set the dial on a hand-checked sample: here the importers whose tax ID matched exactly, so the right taxpayer is known. Accept pairs scoring {threshold} or more.")
scores = informality.probable_scores(importers, registry)
truth = exact[exact["exact_status"] == "matched"].set_index("importer_id")["tin"]
dial = informality.dial(scores[scores["importer_id"].isin(truth.index)], truth, thresholds=range(50, 101, 5)).set_index("threshold")
fig, ax = nb.chart(f"At {threshold}: {dial.at[threshold, 'found']} found, {dial.at[threshold, 'wrong']} wrong, {dial.at[threshold, 'missed']} missed, on {len(truth):,} known pairs")
ax.plot(dial.index, dial["found"], color=nb.COLORS["accent"], marker="o", label="found (the right taxpayer)")
ax.plot(dial.index, dial["wrong"] * 10, color=nb.COLORS["amber"], marker="o", label="wrong (x10 to be visible)")
ax.axvline(threshold, color=nb.COLORS["ink"], linestyle=":"); ax.set_xlabel("threshold"); ax.set_ylabel("importers"); ax.legend(); plt.show()
twenty_five = informality.probable_scores(sheet, registry)
view = sheet[["ref", "importer_id", "tin"]].merge(twenty_five[["importer_id", "matched_tin", "score"]], on="importer_id", how="left").set_index("ref")
view["exact"] = view["importer_id"].map(exact.set_index("importer_id")["exact_status"])
view["accepted"] = view["score"] >= threshold
nb.show(view.sort_values("score"), formats={"score": "{:.0f}"}, highlight=view["accepted"] & (view["exact"] != "matched"), max_rows=25)
off = view[view["exact"] == "TIN one character off"]
blank = view[view["exact"] == "blank TIN"]
nb.finding(f"Strict (95) is for automatic exchange: {dial.at[95, 'wrong']} wrong, {dial.at[95, 'missed']} missed. Balanced (85) is for a case list a "
           f"person checks: {dial.at[85, 'wrong']} wrong, {dial.at[85, 'missed']} missed. Loose (70) is for exploring: {dial.at[70, 'wrong']} wrong. "
           f"On the 25 of the exercise, at {threshold}: {int((off['score'] >= threshold).sum())} of the {len(off)} importers with an ID one character off "
           f"are found by name, phone and director, and {int((blank['score'] >= threshold).sum())} of the {len(blank)} with no ID are matched to a "
           "taxpayer that is not theirs. Acting on a wrong match accuses the wrong taxpayer: hand-check a sample at the dial you choose.")

## 3 · Track B: the same match on a real register

A public company register (3,000 active companies from two towns) stands in for the tax registry; a
customs importer file of 3,200 rows was made from it with real-world damage (suffix dropped, a
previous name, abbreviations, a typo, address variants), plus 150 importers with no counterpart and 50
second spellings. No identifier, no phone, on either side. The recipe: **clean** both sides the same
way → **compare** four fields → let the data give the **weights** → set the **dial** on a hand-checked
sample. Rows are called by their ids; no name is printed.

In [ ]:
#@title 3 · Clean, compare, weights from the data, dial
HINTS = ["Clean both sides the same way.", "A missing field is not a zero.", "The sample sets the dial, not the weights."]

EXACT_CUT = 0.90  #@param {type:"slider", min:0.8, max:0.98, step:0.01}
REVIEW_CUT = 0.70  #@param {type:"slider", min:0.5, max:0.85, step:0.05}
HINT_LEVEL = 0  #@param {type:"slider", min:0, max:3, step:1}
import re
from rapidfuzz import fuzz
nb.steps("Without an identifier, how do we match a customs importer file to a register, and how sure is each link?",
         "Clean both sides the same way: lower case, punctuation out, abbreviations expanded, legal suffixes dropped.",
         "Compare candidate pairs that share a postcode area or the first four letters of the name: name, address (similarity), postcode, town (exact; a missing field is not a zero).",
         "Let the data give the weights: how often each field agrees among matches and among non-matches (estimated by expectation-maximisation, no labels).",
         f"Score every pair, keep the best per customs row, and set the dial: exact at {EXACT_CUT:.2f} and above, review from {REVIEW_CUT:.2f}, no match below.")
try:
    B = load_tables(DATA_URL, ["trackB_registry", "trackB_customs", "match_scores", "trackB_gleif"])
except Exception as error:  # the real-register files are built by data/build_4b_trackB.py and shipped beside the synthetic set
    B = None
    print(f"Track B files are not in this data release ({type(error).__name__}): run data/build_4b_trackB.py and publish them beside the synthetic tables.")
if B is not None:
    reg, cus = B["trackB_registry"].fillna(""), B["trackB_customs"].fillna("")
    SUFFIXES, LONG = {"ltd", "limited", "plc"}, {"&": "and", "co": "company", "st": "street", "rd": "road"}
    def clean(text):
        low = str(text).lower().replace(".", "").replace("&", " & ")
        return " ".join(LONG.get(t, t) for t in re.sub(r"[^a-z0-9&]+", " ", low).split() if t not in SUFFIXES)
    def prepare(frame, name_col):
        postcode = frame["postcode"].str.upper().str.replace(" ", "", regex=False)
        name = frame[name_col].map(clean)
        return frame.assign(name_clean=name, address_clean=frame["address_line"].map(clean), postcode_clean=postcode,
                            town_clean=frame["post_town"].str.upper().str.strip(), outward=postcode.map(lambda p: p[:-3] if len(p) >= 5 else p),
                            prefix=name.str.replace(" ", "", regex=False).str[:4])
    L, R = prepare(cus, "importer_name"), prepare(reg, "legal_name")
    by_outward, by_prefix = R.groupby("outward").indices, R.groupby("prefix").indices
    pairs = pd.DataFrame([(ci, ri) for ci, (o, p) in enumerate(zip(L["outward"], L["prefix"]))
                          for ri in sorted(set(by_outward.get(o, [])) | set(by_prefix.get(p, [])))], columns=["ci", "ri"])
    left, right = L.iloc[pairs["ci"].to_numpy()], R.iloc[pairs["ri"].to_numpy()]
    name_sim = np.array([fuzz.token_set_ratio(a, b) for a, b in zip(left["name_clean"], right["name_clean"])]) / 100
    address_sim = np.array([fuzz.token_sort_ratio(a, b) for a, b in zip(left["address_clean"], right["address_clean"])]) / 100
    def exact_or_missing(a, b):
        return np.where((a == "") | (b == ""), np.nan, (a == b).astype(float))
    gamma = np.column_stack([(name_sim >= 0.85).astype(float), (address_sim >= 0.95).astype(float),
                             exact_or_missing(left["postcode_clean"].to_numpy(), right["postcode_clean"].to_numpy()),
                             exact_or_missing(left["town_clean"].to_numpy(), right["town_clean"].to_numpy())])
    present, g = ~np.isnan(gamma), np.nan_to_num(gamma)
    m, u, p = np.full(4, 0.9), np.full(4, 0.1), 0.05
    for _ in range(20):  # expectation-maximisation: the data says how often each field agrees among matches (m) and non-matches (u)
        log_m = ((g * np.log(m) + (1 - g) * np.log(1 - m)) * present).sum(axis=1)
        log_u = ((g * np.log(u) + (1 - g) * np.log(1 - u)) * present).sum(axis=1)
        likely_m, likely_u = p * np.exp(log_m), (1 - p) * np.exp(log_u)
        post = likely_m / (likely_m + likely_u)
        m = np.clip((post[:, None] * g * present).sum(0) / (post[:, None] * present).sum(0), 1e-4, 1 - 1e-4)
        u = np.clip(((1 - post)[:, None] * g * present).sum(0) / ((1 - post)[:, None] * present).sum(0), 1e-4, 1 - 1e-4)
        p = float(post.mean())
    agree, disagree = np.log2(m / u), np.log2((1 - m) / (1 - u))
    weights = pd.DataFrame({"agrees among matches (m)": m, "agrees among non-matches (u)": u, "weight if it agrees": agree, "weight if it disagrees": disagree},
                           index=["name", "address", "postcode", "town"])
    nb.show(weights, formats={c: "{:.2f}" for c in weights.columns})
    total = ((g * agree + (1 - g) * disagree) * present).sum(axis=1)
    best_w, worst_w = (np.maximum(agree, disagree) * present).sum(axis=1), (np.minimum(agree, disagree) * present).sum(axis=1)
    score = np.clip((total - worst_w) / np.where(best_w > worst_w, best_w - worst_w, 1.0), 0, 1)
    ranked = pairs.assign(score=score, name_sim=name_sim, address_sim=address_sim, tie=name_sim + address_sim)
    best = ranked.sort_values(["ci", "score", "tie", "ri"], ascending=[True, False, False, True]).drop_duplicates("ci").set_index("ci").reindex(range(len(L)))
    links = pd.DataFrame({"cust_id": L["cust_id"].to_numpy(), "reg_id": np.where(best["ri"].notna(), R["reg_id"].to_numpy()[best["ri"].fillna(0).astype(int)], ""),
                          "confidence": best["score"].fillna(0).round(3), "name_sim": best["name_sim"].fillna(0).round(2), "address_sim": best["address_sim"].fillna(0).round(2)})
    links["band"] = np.select([links["confidence"] >= EXACT_CUT, links["confidence"] >= REVIEW_CUT], ["exact", "review"], "no match")
    bands = links["band"].value_counts().reindex(["exact", "review", "no match"], fill_value=0)
    shipped = B["match_scores"].set_index("cust_id")["band"].reindex(links["cust_id"]).to_numpy()
    review = links[links["band"] == "review"].sort_values("confidence")
    least_sure = review.assign(from_cut=(review["confidence"] - REVIEW_CUT).abs()).nsmallest(3, "from_cut")
    display(Markdown(f"**Bands at {EXACT_CUT:.2f} / {REVIEW_CUT:.2f}:** exact {bands['exact']:,} · review {bands['review']:,} · no match {bands['no match']:,} "
                     f"(the shipped match_scores.csv, cut at 0.90 / 0.70, agrees on {(shipped == links['band'].to_numpy()).mean():.0%} of rows)"))
    nb.show(review.head(12).set_index("cust_id")[["reg_id", "confidence", "name_sim", "address_sim", "band"]], formats={"confidence": "{:.2f}"})
    nb.hint(HINT_LEVEL, HINTS, "4b.match.trackB")
    gleif = B["trackB_gleif"]
    linked = int(gleif["lei"].fillna("").ne("").sum())
    nb.finding(f"{len(pairs):,} candidate pairs from {len(cus):,} importer rows against {len(reg):,} register rows. The data gave the weights: the name "
               f"adds {agree[0]:.1f} when it agrees, the address {agree[1]:.1f}, the postcode {agree[2]:.1f}; the town {agree[3]:.1f}, which is nothing "
               f"(everyone shares two towns). At the dial chosen, {bands['exact']:,} links are exact, {bands['review']:,} sit in the review band listed by "
               f"row id above, {bands['no match']:,} have no match; the three least-sure pairs are "
               + ", ".join(f"{r.cust_id}-{r.reg_id} ({r.confidence:.2f})" for r in least_sure.itertuples())
               + f". With an identifier the match is one line: {linked} of the {len(reg):,} register companies hold a Legal Entity Identifier and join exactly on "
               f"the company number; {len(reg) - linked:,} do not. The sample sets the dial, not the weights; never read a company name aloud.")
else:
    nb.finding("Track B needs the real-register files (trackB_registry, trackB_customs, match_scores, trackB_gleif), which this data release does not "
               "carry: the recipe is clean, compare, weights from the data, dial, and the three pairs you are least sure of are listed by row id.")

## 4 · Four buckets

In [ ]:
#@title 4 · Unregistered, not filing, under-reporting, consistent
MARKUP = 1.25  #@param {type:"number"}
PROBABLE_AT = 85  #@param {type:"slider", min:70, max:100, step:5}
nb.steps("Which of the 25 importers does the VAT office need to act on, and how?",
         f"Match each importer: the tax ID when it is in the registry, else the probable match at {PROBABLE_AT} or more, else no match.",
         "Unregistered: no match, or not registered for VAT. Not filing: two or more of the last six returns missing.",
         f"Under-reporting: sales declared below imports x {MARKUP} (the sector mark-up). Consistent: the rest. Each bucket has a lead agency.")
status = exact.set_index("importer_id")["exact_status"]
probable = twenty_five.set_index("importer_id")
matched = [t if status.get(i) == "matched" else (probable.at[i, "matched_tin"] if i in probable.index and probable.at[i, "score"] >= PROBABLE_AT else "")
           for i, t in zip(sheet["importer_id"], sheet["tin"])]
sorted_ = informality.buckets(sheet.assign(matched_tin=matched), registry, returns, periods, markup=MARKUP)
sorted_ = sorted_.merge(sheet[["importer_id", "import_value_12m"]], on="importer_id")
per = sorted_.groupby("bucket").agg(importers=("importer_id", "size"), import_value_12m=("import_value_12m", "sum"))
per = per.reindex(["unregistered", "not filing", "under-reporting", "consistent"]).fillna(0)
lead = {"unregistered": ("tax", "register for VAT; customs shares the import list", "no"),
        "not filing": ("tax", "filing reminder, then an estimated assessment", "no"),
        "under-reporting": ("tax, with customs data", "desk audit of sales against imports", "yes"),
        "consistent": ("none", "keep in the risk model", "no")}
per["lead agency"], per["action"], per["new data-sharing step"] = zip(*[lead[b] for b in per.index])
fig, ax = nb.chart("Imports by bucket, last 12 months (the mechanical sort, before anyone reads the reasons)")
ax.bar(per.index, per["import_value_12m"] / 1e6, color=[nb.COLORS["amber"], nb.COLORS["amber"], nb.COLORS["accent"], nb.COLORS["muted"]])
ax.set_ylabel("LCU million"); plt.show()
nb.show(per, formats={"importers": "{:.0f}", "import_value_12m": "{:,.0f}"})
acted = per.loc[["unregistered", "not filing", "under-reporting"]]
nb.finding(f"The mechanical sort puts {per.at['unregistered', 'importers']:.0f} of the 25 in unregistered, {per.at['not filing', 'importers']:.0f} in not filing, "
           f"{per.at['under-reporting', 'importers']:.0f} in under-reporting and {per.at['consistent', 'importers']:.0f} in consistent "
           f"(LCU {acted['import_value_12m'].sum():,.0f} of imports in the three buckets that call for an action). Tax leads on all three and customs "
           "supplies the import data; sales far below imports has lawful explanations that the registry and the licence tell apart, which is the exercise.")

## 5 · Borrowed names and the five patterns

In [ ]:
#@title 5 · Borrowed names: one tax ID, one address, one broker, many importers
nb.steps("Does anyone lend its tax ID to others?",
         "On the customs register, group importers by the tax ID and the address they share; keep groups of six or more.",
         "For each group: the broker, the phones and the directors they share, the community they form in the network, and the lender's sales against the imports under its ID.")
net = load_json(DATA_URL, "network")
community = {n["id"]: n["community"] for n in net["nodes"]}
clusters = border.shared_address_clusters(importers)
sales = returns[returns["period"].isin(periods[-12:]) & (returns["filed"] == 1)].groupby("tin")["sales_lcu"].sum()
rows = []
for tin, address, size in clusters[["tin", "address_line", "importers"]].itertuples(index=False):
    members = importers[(importers["tin"] == tin) & (importers["address_line"] == address)]
    lender = registry[registry["tin"] == tin]
    rows.append({"tin": tin, "importers": int(size), "broker": members["default_broker_id"].mode().iloc[0], "phones": members["phone"].nunique(),
                 "directors": members["director_name"].nunique(), "communities": members["importer_id"].map(community).nunique(),
                 "imports_under_tin": tf["value_12m_lcu"].reindex(members["importer_id"]).sum(), "lender_sales_12m": float(sales.get(tin, 0))})
table = pd.DataFrame(rows).set_index("tin")
nb.show(table, formats={"imports_under_tin": "{:,.0f}", "lender_sales_12m": "{:,.0f}"})
top2 = table.head(2)
nb.finding(f"Tax IDs {top2.index[0]} and {top2.index[1]} each cover {top2['importers'].iloc[0]} and {top2['importers'].iloc[1]} importers at one address, "
           f"through one broker each ({top2['broker'].iloc[0]}, {top2['broker'].iloc[1]}) and a handful of phones and directors; the lenders declare sales worth "
           f"{(top2['lender_sales_12m'] / top2['imports_under_tin']).iloc[0]:.0%} and {(top2['lender_sales_12m'] / top2['imports_under_tin']).iloc[1]:.0%} of the imports "
           "under their ID. A borrowed name moves the liability to a shell; a group clearing for its own subsidiaries has one address each. Customs and tax visit together.")

In [ ]:
#@title 5 · Pattern 1: parcels split under the duty-free limit
split = border.parcel_splitting(T["parcels"])
repeat = split[split["months_flagged"] >= 3]
nb.show(split.head(6), formats={"max_value_lcu": "{:,.0f}"})
nb.finding(f"{len(repeat)} receivers get six parcels or more a month just under the limit in at least three months; neither duty nor VAT is paid on any of "
           "them. Customs acts: aggregate parcels by receiver before applying the limit.")

In [ ]:
#@title 5 · Pattern 2: the land border, crossing by crossing
lbm = T["land_border_mirror"].assign(ratio=lambda x: x["home_imports_lcu"] / x["neighbour_exports_lcu"])
fig, ax = nb.chart("Home imports against the neighbour's exports, by crossing")
for office, part in lbm.groupby("crossing_office"):
    part = part.sort_values("month")
    ax.plot(part["month"], part["ratio"], label=f"{office} ({part['neighbour'].iloc[0]})")
ax.axhline(1.0, color=nb.COLORS["muted"], linestyle=":"); ax.set_xticks(sorted(lbm["month"].unique())[::6]); ax.legend(fontsize=8); plt.show()
trend = {o: np.polyfit(range(len(p)), p.sort_values("month")["ratio"], 1)[0] for o, p in lbm.groupby("crossing_office")}
worst = min(trend, key=trend.get)
part = lbm[lbm["crossing_office"] == worst].sort_values("month")
nb.finding(f"At {worst} home imports fall from {part['ratio'].iloc[0]:.0%} to {part['ratio'].iloc[-1]:.0%} of what {part['neighbour'].iloc[0]} records as exported "
           f"through it; the other {len(trend) - 1} crossings stay near parity. Goods crossing unrecorded are counted at the crossing: joint checks with the neighbour, "
           "and a survey post at the crossing (direct observation, two weeks a month, is how one administration measures its informal trade).")

In [ ]:
#@title 5 · Pattern 3: a sector that sells less than it imports
frame = importers.assign(sales=importers["tin"].map(sales), imports=tf["value_12m_lcu"].reindex(importers["importer_id"]).to_numpy()).dropna(subset=["sales"])
frame = frame[frame["imports"] > 0]
share = (frame["sales"] < frame["imports"]).groupby(frame["sector"]).mean().sort_values(ascending=False)
fig, ax = nb.chart(f"{share.index[0]}: {share.iloc[0]:.0%} of importers declare sales below their imports")
ax.barh(share.index[::-1], share.values[::-1] * 100, color=[nb.COLORS["amber"] if s == share.index[0] else nb.COLORS["accent"] for s in share.index[::-1]])
ax.set_xlabel("importers selling less than they import, %"); plt.show()
nb.finding(f"In {share.index[0]}, {share.iloc[0]:.0%} of importers declare sales below their imports, against {share.drop(share.index[0]).median():.0%} in the "
           "median sector: a sector that sells in cash under-declares sales, not imports. Tax acts, with customs import data, as a sector campaign.")

In [ ]:
#@title 5 · Pattern 4: origin washed through one seller
producers = T["tariff"].set_index("hs8")["origin_producers"].str.split("|")
claims = f[(f["preference_claim"] == "FTA") & (f["exporter_id"] != "") & f["regime"].isin(HOME_USE)]
makes = np.array([o in producers.get(c, []) for c, o in zip(claims["hs8"], claims["origin"])])
doubtful = claims[~makes]
sellers = doubtful.groupby("exporter_id").agg(importers=("importer_id", "nunique"), lines=("line_id", "size"),
                                              before_month_30=("period_month", lambda s: int((s <= 30).sum())), after_month_30=("period_month", lambda s: int((s > 30).sum())))
sellers = sellers.sort_values("importers", ascending=False)
nb.show(sellers.head(5))
top = sellers.head(5)
nb.finding(f"Sellers {sellers.index[0]} and {sellers.index[1]} are behind most claims from origins that do not make the goods ({sellers['importers'].iloc[0]} and "
           f"{sellers['importers'].iloc[1]} importers). For the five widest sellers {top['after_month_30'].sum()} of their {top['lines'].sum()} doubtful lines fall "
           "after month 30: a scheme that changes its paperwork keeps its sellers. Customs acts: verify the certificates with the issuing authority.")

In [ ]:
#@title 5 · Pattern 5: one tax ID, many names
per_id = importers[importers["tin"] != ""].groupby("tin").agg(importers=("importer_id", "size"), addresses=("address_line", "nunique"), brokers=("default_broker_id", "nunique"))
per_id["importers_per_address"] = (per_id["importers"] / per_id["addresses"]).round(1)
per_id = per_id[per_id["importers"] >= 2].sort_values(["importers_per_address", "importers"], ascending=False)
nb.show(per_id.head(5))
nb.finding(f"Tax IDs {per_id.index[0]} and {per_id.index[1]} clear for {per_id['importers_per_address'].iloc[0]:.0f} and {per_id['importers_per_address'].iloc[1]:.0f} "
           f"importers per address through {per_id['brokers'].iloc[0]} broker each; a group clearing for its own subsidiaries has one address each. Both sides act: a joint visit.")

In [ ]:
#@title 5 · Framing: how much of the region's employment is informal
SHARES = [7.3, 20.1, 29.1, 32.6, 33.4, 35.3, 37.2, 40.2, 45.7, 47.3, 51.5, 59.2, 61.5, 63.2, 66.4, 67.0, 70.8, 78.5, 80.6, 80.7, 80.9, 81.5, 84.0, 86.1, 87.2, 88.7, 90.5]

nb.steps("How large is informality in the region, as an open statistic?",
         "The ILO's SDG 8.3.1 series: informal employment as a share of total employment, latest national survey, one dot per economy, none named.")
if SHARES:
    values = np.array(SHARES)
    fig, ax = nb.chart(f"Informal employment in {len(values)} economies of the region: median {np.median(values):.0f}%", height=2.6)
    jitter = np.random.default_rng(3).uniform(-0.18, 0.18, len(values))
    ax.scatter(values, jitter, s=90, color=nb.COLORS["accent"], alpha=0.9, zorder=3)
    ax.axvline(np.median(values), color=nb.COLORS["amber"], linewidth=2.5)
    ax.set_xlim(0, 100); ax.set_ylim(-0.5, 0.5); ax.set_yticks([]); ax.set_xlabel("informal employment as a share of total employment, %"); plt.show()
    nb.finding(f"Across {len(values)} economies of the region informal employment runs from {values.min():.0f}% to {values.max():.0f}% of employment, median "
               f"{np.median(values):.0f}% (ILO SDG 8.3.1, latest survey per economy; the economies are unnamed on purpose). The importers without a tax ID "
               "in section 1 are the border's share of this.")
else:
    nb.finding("The ILO series was not available when this notebook was built: the framing exhibit is in the deck (27 economies, 7 to 90%, median 63%).")

## 6 · Exercise: 25 importers, two tracks

**Track A.** Roles: the **analyst** reads the table, the **officer** sorts each importer into a bucket and
chooses the action and the lead agency, the **committee** marks the actions that need a new data-sharing
step. Then name the two networks by their lead tax IDs. **Track B.** Match the customs importer file to the
register (section 3): write the dial you chose and why, and the three pairs you are least sure of, by row id.
Nothing about the answer is visible until the reveal.

In [ ]:
#@title 6 · The 25 importers
HINTS = ["Start from what the registry and the returns say.", "Match, then registration, filing, and sales against imports; then look at shared addresses and brokers.", "Under-reporting includes the two lenders and the one-digit TIN trap."]

HINT_LEVEL = 0  #@param {type:"slider", min:0, max:3, step:1}
recent = returns[returns["period"].isin(periods[-12:])]
filed = recent[recent["filed"] == 1].groupby("tin").size()
ws = sheet.set_index("ref")
ws["exact_match"] = ws["importer_id"].map(status).fillna("")
ws["returns_filed_12m"] = ws["tin"].map(filed).fillna(0).astype(int)
ws["sales_declared_12m"] = ws["tin"].map(sales).fillna(0)
nb.show(ws[["importer_id", "tin", "exact_match", "sector", "consignees", "import_value_12m", "returns_filed_12m", "sales_declared_12m"]],
        formats={"import_value_12m": "{:,.0f}", "sales_declared_12m": "{:,.0f}"}, max_rows=25)
nb.hint(HINT_LEVEL, HINTS, "4b.exercise")
nb.finding(f"{int((ws['exact_match'] != 'matched').sum())} of the 25 have no exact registry match and {int((ws['returns_filed_12m'] < 12).sum())} filed fewer "
           "than twelve returns: start from those, then compare sales with imports, then look at shared addresses and brokers. A probable match is not a match: ask for its confidence.")

In [ ]:
#@title 6 · Your buckets, actions, leads, and track B
import ipywidgets as widgets
BUCKETS, ACTIONS = ['', 'unregistered', 'not filing', 'under-reporting', 'consistent'], {'': ['', False], 'register for VAT': ['tax', False], 'filing reminder, then estimated assessment': ['tax', False], 'desk audit: sales against imports': ['tax', True], 'valuation query or post-clearance audit': ['customs', False], 'joint visit': ['customs and tax', True], 'no action: keep in the risk model': ['', False]}
rows = [widgets.HBox([widgets.Label(i, layout=widgets.Layout(width="90px")),
                      widgets.Dropdown(options=BUCKETS, layout=widgets.Layout(width="140px")),
                      widgets.Dropdown(options=list(ACTIONS), layout=widgets.Layout(width="280px")),
                      widgets.Dropdown(options=["", "customs", "tax", "customs and tax"], layout=widgets.Layout(width="140px")),
                      widgets.Checkbox(description="new data-sharing step", indent=False)]) for i in ws.index]
networks = widgets.Text(description="Networks", placeholder="the two lead tax IDs, comma separated")
dial_b = widgets.Text(description="Track B dial", placeholder="the cut you chose, and why (one line)")
pairs_b = widgets.Text(description="Least sure", placeholder="three pairs by row id, e.g. C00123-R00456")
lock = widgets.Button(description="Lock my answers", button_style="primary")
note = widgets.Output()
LOCKED = {"buckets": {}, "actions": {}, "networks": [], "track_b": {}}
def on_lock(_):
    LOCKED["buckets"] = {r.children[0].value: r.children[1].value for r in rows}
    LOCKED["actions"] = {r.children[0].value: r.children[2].value for r in rows}
    LOCKED["networks"] = [t.strip() for t in networks.value.split(",") if t.strip()]
    LOCKED["track_b"] = {"dial": dial_b.value, "least_sure": pairs_b.value}
    with note:
        note.clear_output()
        print(f"Locked: {sum(1 for v in LOCKED['buckets'].values() if v)} of 25 importers bucketed; networks: "
              f"{', '.join(LOCKED['networks']) or 'none'}; track B dial: {dial_b.value or 'not written'}.")
lock.on_click(on_lock)
display(widgets.VBox(rows + [networks, dial_b, pairs_b, lock, note]))
#@markdown If the menus do not show, type "I01: unregistered, I02: consistent, ..." and the two networks here instead.
MY_BUCKETS = ""  #@param {type:"string"}
MY_NETWORKS = ""  #@param {type:"string"}
if MY_BUCKETS or MY_NETWORKS:
    LOCKED["buckets"] = dict(p.split(":", 1) for p in MY_BUCKETS.replace(" ", "").split(",") if ":" in p)
    LOCKED["networks"] = [t.strip() for t in MY_NETWORKS.split(",") if t.strip()]
    print(f"Typed in: {len(LOCKED['buckets'])} buckets; networks {', '.join(LOCKED['networks'])}.")

## Reveal (facilitator)

The facilitator types the code. Without it, this cell prints nothing.

In [ ]:
#@title 6 · Reveal
REVEAL_CODE = ""  #@param {type:"string"}
SEALED = {"salt": "I15g1d/M7TxrrY4zViT7uQ==", "data": "LS+mCJbZpUpyCkZ6dXe4zYU+wE760kI1ZG+MTjXOSPRL4FYPoD79JolpVq25ut1YBr/BX3f/yN5auFsJfn7Jvy/49AeDfVwFjxyOnM60If+AeoKsu2HGzIHiTTYcPS49o+JE3uHtkr3wjzcKpcoX3rlJjWgyWxIIGW1NoQQAI3mssDNUNYVPiFBWGzVDrODG6PS+IdC/BFxiWUGmzAbGqJBxpnj95DYtIeePRKZ4RmIlQ7rpgBQkVMH8dFijim0Z93W/EF5ge5BfvvanBADyzs8lKdIbXFZR63Bp1PZP/o098Sycc8+Pm5kv+b1OVD839rbb1d2ryVOETFwt7eMNC64IQ9AtGP+m//ZKaAGxGFz80LH1ObuDa3eiwe1kAiAZxk+TOazfpV8Zj9M2Ho6lAyPpant+4oveIiM+8Dxr1A3pI+eBIaLsiUAwquQ17fcH+jEYOaP1J4GFXKDZYSIPrrr5sT4EXuCuCHwGI7g6QczeflKb7jl5AZYnAeXzhMgI+QAGO/jx+YBAnD9XLCebIqktf2NEx2Ioa8kBbppisoSSYrmlQ4OLqLZi/2DJvV5z5ESRgajts5hZvPLhgJo4KqXcY0abUsM+FcQJJIbRB0bKazQLIzMJugJOul1dbq2hqON59FIECk+OBOx9qG9vq2rB7SeJJddxMoKHep6nZZCCliEZJa1PIIx0dEB8N8e/kUMWdZF5fNI+UPq0eNvzqZY/A3P9n02IXnZW6I7Obs19hOQhUOll0Y58IJ249JHa6rf+mlFTrBGia9tVQkN11gOvj8/2AT7Cyx0t+7Skv24JjF3p/dV2nVks+vdJssGTDVxSqNclFvwbe3uCsVf5z+NkzhqXcm3jo19TiXnK0yt3HeV+cbUnMT8lfW/NnuMKg357i6UEaau+zv3P9zRfmphrr+4knSZx5PJycmhPDDVBFuLppt0MDGArf+45jug3OrYxwOX2Qw7vtxLOg+yHRO7aQcC6wCnTzAoTA+ds6jFzSjUGupywm3yGIUSizkSNERcWQ4aVqOZ08dtBM+po8PN/A6Wyx4QYsQQ9+x3wZGzZNjUs/aPtdFa6n/MUU+qI1+o8B0LYiSWpscUNHTKRIo4CtyPX/trVVypIBUyHcnhwOl+fFGMQvCYZc1kCgA2V0M1yKTjI15LCdl71OTKkBO6KJHdlaMzvg6lUBqW2CAJIWtTSLOUR9ZI4ElsuF9jPRyJzM0pZCXDbNCSSkvSArOTBy4EnHechGHk9xVw12uCkW2ZDpmYMmJ9dKtSM20WSnkoHI0IXsr45qUdbUpsCd1qO1dxsC7L3fbbaysMFsN3rUX4//Yvh1tuuAmKnMXeIrak/h8FClAe3j/gVNd/vlk92isAPzaIg5S3YmefvicrFEanv", "mac": "0e6400c964b301c8b8041b85611a03fd5dee59eddcb47b7783211012fe962b1c"}
answer = nb.unseal(REVEAL_CODE, SEALED)
if answer:
    key = pd.DataFrame(answer["rows"]).set_index("ref")
    key["your bucket"] = key.index.map(lambda i: LOCKED["buckets"].get(i, "").replace("-", " ").replace("_", " ").strip())
    key["right"] = key["your bucket"] == key["bucket"].str.replace("-", " ")
    key["import_value_12m"] = ws["import_value_12m"].reindex(key.index)
    nb.show(key[["importer", "bucket", "your bucket", "right", "why"]], max_rows=25, highlight=~key["right"])
    per = key.groupby("bucket")["import_value_12m"].agg(["size", "sum"]).rename(columns={"size": "importers", "sum": "import_value_12m"})
    nb.show(per, formats={"import_value_12m": "{:,.0f}"})
    nets = pd.DataFrame(answer["networks"])
    found = len(set(LOCKED["networks"]) & set(nets["lead_tin"]))
    chosen = [a for a in LOCKED["actions"].values() if a]
    today = sum(1 for a in chosen if not ACTIONS[a][1])
    trap = answer["identifier_trap"]
    nb.finding(f"You bucketed {int(key['right'].sum())} of 25 importers right and found {found} of the 2 networks "
               f"({'; '.join(f'{r.lead_tin}: {r.consignees} importers at {r.shared_address}, broker {r.shared_broker}' for r in nets.itertuples())}). "
               f"The mistyped tax ID: customs holds {trap['customs_tin']} where the registry holds {trap['registry_tin']}. "
               f"{today} of your {len(chosen)} actions are possible today without a new agreement.")
    if "track_b" in answer:
        tb = answer["track_b"]
        print(f"Track B: bands exact {tb['bands']['exact']}, review {tb['bands']['review']}, no match {tb['bands']['no match']}; precision in the exact band "
              f"{tb['precision_exact']:.1%}; at 0.70 recall {tb['recall_at_0.70']:.0%} and precision {tb['precision_at_0.70']:.0%}; {tb['orphans']} importers had no "
              f"counterpart and {tb['duplicates']} were second spellings; {tb['gleif_linked']} register companies joined on an identifier in one line. "
              f"Your dial: {LOCKED['track_b'].get('dial') or 'not written'}.")
    for item in answer["explanations"]:
        print(f"- {item}")
    if nb.HINTS_USED:
        print("Hints used:", nb.HINTS_USED)

**Thursday homework.** Download one open statistics file of your own administration (monthly imports by tariff line and partner, or declarations by office) and name the field you would add to it first.

## 7 · For your analysts

**Deterministic and probabilistic linkage.** An exact join on the tax ID is deterministic: it is right when
it matches and silent when it does not. Probable matching scores every candidate pair and accepts those above
a threshold; it finds what the ID misses and sometimes accepts the wrong taxpayer.

**Blocking.** Comparing every customs record with every registry record is slow; compare only pairs that share
something cheap to check (the first letters of the cleaned name and the district on the synthetic set; the
postcode area or the first four letters of the name on the real register).

**Weights from the data.** On track B the weights are not set by hand: the Fellegi-Sunter model estimates, by
expectation-maximisation, how often each field agrees among matches and among non-matches, and the weight of a
field is the log of that ratio. A missing field drops out of the likelihood; it is not a disagreement. The
hand-labelled sample sets the dial (precision and recall at each cut), never the weights. Splink is the
open-source tool most administrations use in production.

**Choosing the threshold.** Hand-check a random sample of pairs at a few thresholds and plot found against
wrong. Strict thresholds suit automatic exchange; balanced ones suit case lists a person checks; the review
band is where the dial matters, listed by row id.

**The identifier.** With a shared identifier the match is one query (the GLEIF join on the company number);
without it, it is a dial. The cheapest reform in this session is a check digit on the tax ID at registration.

**Governance checklist.** A shared identifier and its validation; outcomes recorded on both sides; a named
owner for the match; the legal basis for the exchange; a fairness check (who is matched wrongly most often);
privacy (minimum fields, retention, access log); no real name read aloud from a register.

In [ ]:
#@title 7 · Matching in a few lines
exact = informality.exact_match(importers, registry)                      # deterministic: join on the tax ID
scores = informality.probable_scores(importers, registry)                 # probabilistic: name, address, phone, director
accepted = scores[scores["score"] >= 85]                                  # the dial
print(exact["exact_status"].value_counts().to_string(), f"\nprobable matches at 85: {len(accepted)}")